In [ ]:

from IPython import get_ipython
from IPython.display import display

!pip install --upgrade transformers datasets accelerate

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration, Trainer, TrainingArguments
import numpy as np
import nltk
from datasets import load_metric # datasets 라이브러리에서 load_metric 함수 가져오기

# 📁 2. 데이터 업로드
from google.colab import files
uploaded = files.upload()

# 1. 데이터 로딩 및 전처리
df = pd.read_csv('call_qa_1_1000.csv', encoding='cp949')
df = df[['고객질문(공격)', '고객질문(요청)']].dropna()
df.columns = ['input', 'target']

# 2. 토크나이저 & 모델 불러오기
model_name = "beomi/kcbert-base"  # KcBART는 beomi/kcbert-base 기반으로 커스텀할 수 있음
tokenizer = PreTrainedTokenizerFast.from_pretrained("gogamza/kobart-base-v2")
model = BartForConditionalGeneration.from_pretrained("gogamza/kobart-base-v2")

# 3. Dataset 정의
class HateSpeechDataset(Dataset):
    def __init__(self, inputs, targets, tokenizer, max_len=128):
        self.inputs = inputs
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input = self.inputs[idx]
        target = self.targets[idx]
        inputs = self.tokenizer(
            input, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt"
        )
        targets = self.tokenizer(
            target, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt"
        )

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": targets["input_ids"].squeeze(),
        }

# 4. Train/Val 분할
from sklearn.model_selection import train_test_split

train_inputs, val_inputs, train_targets, val_targets = train_test_split(
    df['input'].tolist(), df['target'].tolist(), test_size=0.2, random_state=42
)

train_dataset = HateSpeechDataset(train_inputs, train_targets, tokenizer)
val_dataset = HateSpeechDataset(val_inputs, val_targets, tokenizer)




In [ ]:
# 5. 학습 설정
training_args = TrainingArguments(
    output_dir="./kcbert_paraphrase",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    eval_strategy="epoch",  # evaluation_strategy를 eval_strategy로 변경
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none" # WandB 로깅 비활성화
)



# 6. Trainer 정의 및 학습
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

pred_ids = [[1, 456, 789, 2, 1, 1, 1]]  # 예시: <s> ... </s>
decoded = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
print(decoded)

def generate_paraphrase(text):
    input_ids = tokenizer(text, return_tensors="pt", padding=True, truncation=True).input_ids
    output_ids = model.generate(input_ids, max_length=50, num_beams=4, early_stopping=True)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 예시 문장
offensive = "야 너 진짜 뇌 없냐?"
print("순화 표현:", generate_paraphrase(offensive))


Epoch,Training Loss,Validation Loss
1,9.265100,3.335944


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,9.265100,3.335944
2,0.358100,0.184398


In [ ]:
# 예시 문장
offensive = "그렇게 말씀하실 문제가 맞아요?"
print("순화 표현:", generate_paraphrase(offensive))

NameError: name 'generate_paraphrase' is not defined

In [ ]:
# ⚙️ Colab 환경에서 필요한 패키지 설치
!pip install --upgrade transformers datasets accelerate

# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("summarization", model="eenzeenee/t5-base-korean-summarization")

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from google.colab import files

# 📁 데이터 업로드
uploaded = files.upload()  # 'call_qa_1_1000.csv' 파일 업로드

# 📊 데이터 전처리
df = pd.read_csv('call_qa_1_1000.csv', encoding='cp949')
df = df[['고객질문(공격)', '고객질문(요청)']].dropna()
df.columns = ['input', 'target']

# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("eenzeenee/t5-base-korean-summarization")
model = AutoModelForSeq2SeqLM.from_pretrained("eenzeenee/t5-base-korean-summarization")

# 🧱 Dataset 클래스 정의
class ParaphraseDataset(Dataset):
    def __init__(self, inputs, targets, tokenizer, max_len=128):
        self.inputs = inputs
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input = self.inputs[idx]
        target = self.targets[idx]
        inputs = self.tokenizer(
            input, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt"
        )
        targets = self.tokenizer(
            target, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt"
        )

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": targets["input_ids"].squeeze(),
        }

# 📦 학습/검증 데이터 분리
train_inputs, val_inputs, train_targets, val_targets = train_test_split(
    df['input'].tolist(), df['target'].tolist(), test_size=0.2, random_state=42
)
train_dataset = ParaphraseDataset(train_inputs, train_targets, tokenizer)
val_dataset = ParaphraseDataset(val_inputs, val_targets, tokenizer)

# ⚙️ 학습 설정
training_args = TrainingArguments(
    output_dir="./krbart_paraphrase",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    eval_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

# 🧠 Trainer 정의 및 학습
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

# ✅ 추론 함수 정의
def generate_paraphrase(text):
    input_ids = tokenizer(text, return_tensors="pt", padding=True, truncation=True).input_ids
    output_ids = model.generate(
        input_ids,
        max_length=50,
        num_beams=5,
        repetition_penalty=2.5,
        early_stopping=True
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 📌 테스트
example_text = "야 너 진짜 뇌 없냐?"
print("원문:", example_text)
print("순화 표현:", generate_paraphrase(example_text))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.41k [00:00<?, ?B/s]

...

Saving call_qa_1_1000.csv to call_qa_1_1000.csv


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,9.265100,3.335944
2,0.358100,0.184398
3,0.123200,0.106244
4,0.094400,0.098883


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,9.265100,3.335944
2,0.358100,0.184398
3,0.123200,0.106244
4,0.094400,0.098883


In [ ]:
example_text = "개씨발 애미 뒤진 샛기"
print("원문:", example_text)
print("순화 표현:", generate_paraphrase(example_text))

원문: 개씨발 애미 뒤진 샛기
순화 표현: 애미 뒤진 샛기 시발 개씨발 애미를 뒤흔든 새기는 개년이다.
